In [231]:
import jax.numpy as jnp
import jax.lax as lax
from jax import jit
from functools import partial
from matplotlib import pyplot as plt
import diffrax as dif
import pandas as pd

# Plot-Formatierung
plt.rcParams['font.size'] = 24.0
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = 'Arial'
plt.rcParams['font.weight'] = 'bold'
plt.rcParams['axes.labelsize'] = 'medium'
plt.rcParams['axes.labelweight'] = 'bold'
plt.rcParams['axes.linewidth'] = 1.2
plt.rcParams['lines.linewidth'] = 2.0

#Formulierung des Anfangswertproblems
f = lambda params, z, u: jnp.array([u[1], params * u[0] + z*1e-4])

z0 = 0.
z1 = 12.
u0 = jnp.array([1., -1e-2])
params = 0.01

In [295]:
@partial(jit, static_argnames=['f', 'n'])
def eigenerSolverV2(params, z0, z1, u0, f, n):
    dz = (z1-z0)/(n-1)

    # Runge-Kutta 4. Ordnung
    def rk4_step(params, z0, dz, u0, f):
        k1 = dz * f(params, z0, u0)
        k2 = dz * f(params, z0 + dz / 2, u0 + k1 / 2)
        k3 = dz * f(params, z0 + dz / 2, u0 + k2 / 2)
        k4 = dz * f(params, z0 + dz, u0 + k3)
        u1 = u0 + (k1 + 2 * k2 + 2 * k3 + k4) / 6
        return u1

    def rk4_step_scan(u, x):
        return rk4_step(params, x, dz, u, f), \
            rk4_step(params, x, dz, u, f)

    zs = jnp.linspace(z0, z1, n)
    _, uz = lax.scan(rk4_step_scan, u0, zs[:-1])
    uz = jnp.concatenate([jnp.array([u0]), uz], axis=0)

    return uz, zs

@partial(jit, static_argnames=['f', 'n', 'rtol', 'atol']) 
def diffraxDopri5(params, z0, z1, u0, f, n, rtol, atol):

    vector_field = lambda z, y, args: f(args, z, y) #wrapper für reihenfolge
    term = dif.ODETerm(vector_field)
    solver = dif.Dopri5()
    saveat = dif.SaveAt(steps=True)#ts=jnp.linspace(0, z1, n))
    stepsize_controller = dif.PIDController(rtol=rtol, atol=atol)

    sol = dif.diffeqsolve(  term, solver, 
                            t0=z0, t1=z1, dt0=None, y0=u0, args=(params), 
                            saveat=saveat,
                            stepsize_controller=stepsize_controller)
                            #max_steps=None)

    zs = sol.ts
    uz = sol.ys

    return uz, zs

f_analytical = lambda z: -1e-2*z + jnp.cosh(0.1*z)

In [296]:
ns = [5, 10, 20, 40, 60]

df = pd.DataFrame(columns=['n', 'abw_z1_eigen', 'abw_z1_dopri', 'abw_mean_eigen', 'abw_mean_dopri'])

for n in ns:
    uz_eigen, zs_eigen = eigenerSolverV2(params, z0, z1, u0, f, n)
    uz_dopri, zs_dopri = diffraxDopri5(params, z0, z1, u0, f, n, 1e-3, 1e-6)

    uz_dopri = uz_dopri[~jnp.isinf(uz_dopri).any(axis=1)]
    zs_dopri = zs_dopri[~jnp.isinf(zs_dopri)]

    abw_z1_eigen = uz_eigen[-1,0] - f_analytical(zs_eigen[-1])
    abw_z1_dopri = uz_dopri[-1,0] - f_analytical(zs_dopri[-1])
    abw_mean_eigen = jnp.mean(jnp.abs(uz_eigen[:,0] - f_analytical(zs_eigen)))
    abw_mean_dopri = jnp.mean(jnp.abs(uz_dopri[:,0] - f_analytical(zs_dopri)))

    df = pd.concat([df, pd.DataFrame({'n': [n], 'abw_z1_eigen': [abw_z1_eigen], 'abw_z1_dopri': [abw_z1_dopri], 'abw_mean_eigen': [abw_mean_eigen], 'abw_mean_dopri': [abw_mean_dopri]})], ignore_index=True)

    # # Plot
    # fig, ax1 = plt.subplots(figsize=(20,10))
    # plt.title('Test')
    # ax1.set_xlabel('z/pc')
    # ax1.set_ylabel('$\Phi / (km/s)^2$')
    # ax1.scatter(zs_eigen, [u[0] for u in uz_eigen], marker='o', color = 'black', label = 'eigenerSolver')
    # ax1.scatter(zs_dopri, [u[0] for u in uz_dopri], marker='o', color = 'red', label = 'Dopri5')
    # zs = jnp.linspace(z0, z1, 100)
    # ax1.plot(zs, f_analytical(zs), color = 'blue', label = 'Analytische Lösung')
    # ax1.grid()
    # ax1.legend()
    # fig.tight_layout()

df.to_string('StepSize.txt', col_space=16, justify='left', index=False)

In [297]:
n = 4
rtols = [1e-1, 1e-3, 1e-6, 1e-9]
atol = 0

df = pd.DataFrame(columns=['rtol', 'abw_eigen', 'abw_dopri', 'abw_mean_eigen', 'abw_mean_dopri'])

for rtol in rtols:

    uz_eigen, zs_eigen = eigenerSolverV2(params, z0, z1, u0, f, n)
    uz_dopri, zs_dopri = diffraxDopri5(params, z0, z1, u0, f, n, rtol, atol)

    uz_dopri = uz_dopri[~jnp.isinf(uz_dopri).any(axis=1)]
    zs_dopri = zs_dopri[~jnp.isinf(zs_dopri)]

    abw_eigen = jnp.array([uz_eigen[i,0] - f_analytical(zs_eigen[i]) for i in range(n)])
    abw_dopri = jnp.array([uz_dopri[i,0] - f_analytical(zs_dopri[i]) for i in range(len(zs_dopri))])
    abw_mean_eigen = jnp.mean(jnp.abs(abw_eigen))
    abw_mean_dopri = jnp.mean(jnp.abs(abw_dopri))

    df = pd.concat([df, pd.DataFrame({'rtol': [rtol], 'abw_eigen': [tuple(abw_eigen.tolist())], 'abw_dopri': [tuple(abw_dopri.tolist())], 'abw_mean_eigen': [abw_mean_eigen], 'abw_mean_dopri': [abw_mean_dopri]})], ignore_index=True)

    # Plot
    # fig, ax1 = plt.subplots(figsize=(20,10))
    # plt.title('Test')
    # ax1.set_xlabel('z/pc')
    # ax1.set_ylabel('$\Phi / (km/s)^2$')
    # ax1.scatter(zs_eigen, [u[0] for u in uz_eigen], marker='o', color = 'black', label = 'eigenerSolver')
    # ax1.scatter(zs_dopri, [u[0] for u in uz_dopri], marker='o', color = 'red', label = 'Dopri5')
    # zs = jnp.linspace(z0, z1, 100)
    # ax1.plot(zs, f_analytical(zs), color = 'blue', label = 'Analytische Lösung')
    # ax1.grid()
    # ax1.legend()
    # fig.tight_layout()

df.to_csv('rtol.csv', sep=',', index=False)

C:\Users\Ruben\AppData\Local\Temp\ipykernel_35996\413142441.py:20: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame({'rtol': [rtol], 'abw_eigen': [tuple(abw_eigen.tolist())], 'abw_dopri': [tuple(abw_dopri.tolist())], 'abw_mean_eigen': [abw_mean_eigen], 'abw_mean_dopri': [abw_mean_dopri]})], ignore_index=True)


In [298]:
n = 4
rtol = 0
atols = [1e-1, 1e-3, 1e-6, 1e-9]

df = pd.DataFrame(columns=['atol', 'abw_eigen', 'abw_dopri', 'abw_mean_eigen', 'abw_mean_dopri'])

for atol in atols:

    uz_eigen, zs_eigen = eigenerSolverV2(params, z0, z1, u0, f, n)
    uz_dopri, zs_dopri = diffraxDopri5(params, z0, z1, u0, f, n, rtol, atol)

    uz_dopri = uz_dopri[~jnp.isinf(uz_dopri).any(axis=1)]
    zs_dopri = zs_dopri[~jnp.isinf(zs_dopri)]

    abw_eigen = jnp.array([uz_eigen[i,0] - f_analytical(zs_eigen[i]) for i in range(n)])
    abw_dopri = jnp.array([uz_dopri[i,0] - f_analytical(zs_dopri[i]) for i in range(len(zs_dopri))])
    abw_mean_eigen = jnp.mean(jnp.abs(abw_eigen))
    abw_mean_dopri = jnp.mean(jnp.abs(abw_dopri))

    df = pd.concat([df, pd.DataFrame({'atol': [atol], 'abw_eigen': [tuple(abw_eigen.tolist())], 'abw_dopri': [tuple(abw_dopri.tolist())], 'abw_mean_eigen': [abw_mean_eigen], 'abw_mean_dopri': [abw_mean_dopri]})], ignore_index=True)

    # Plot
    # fig, ax1 = plt.subplots(figsize=(20,10))
    # plt.title('Test')
    # ax1.set_xlabel('z/pc')
    # ax1.set_ylabel('$\Phi / (km/s)^2$')
    # ax1.scatter(zs_eigen, [u[0] for u in uz_eigen], marker='o', color = 'black', label = 'eigenerSolver')
    # ax1.scatter(zs_dopri, [u[0] for u in uz_dopri], marker='o', color = 'red', label = 'Dopri5')
    # zs = jnp.linspace(z0, z1, 100)
    # ax1.plot(zs, f_analytical(zs), color = 'blue', label = 'Analytische Lösung')
    # ax1.grid()
    # ax1.legend()
    # fig.tight_layout()

df.to_csv('atol.csv', sep=',', index=False)

C:\Users\Ruben\AppData\Local\Temp\ipykernel_35996\1629720778.py:20: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame({'atol': [atol], 'abw_eigen': [tuple(abw_eigen.tolist())], 'abw_dopri': [tuple(abw_dopri.tolist())], 'abw_mean_eigen': [abw_mean_eigen], 'abw_mean_dopri': [abw_mean_dopri]})], ignore_index=True)


In [300]:
n = 11
rtol = 0
atol = 1e-6

df = pd.DataFrame(columns=['abw_mean_eigen', 'abw_mean_dopri', 'abw_max_eigen', 'abw_max_dopri'])

print('eigenerSolverV2')
%time eigenerSolverV2(params, z0, z1, u0, f, n).block_until_ready()
%timeit eigenerSolverV2(params, z0, z1, u0, f, n).block_until_ready()

print('Dopri5')
%time diffraxDopri5(params, z0, z1, u0, f, n, rtol, atol)
%timeit diffraxDopri5(params, z0, z1, u0, f, n, rtol, atol)

uz_eigen, zs_eigen = eigenerSolverV2(params, z0, z1, u0, f, n)
uz_dopri, zs_dopri = diffraxDopri5(params, z0, z1, u0, f, n, rtol, atol)

uz_dopri = uz_dopri[~jnp.isinf(uz_dopri).any(axis=1)]
zs_dopri = zs_dopri[~jnp.isinf(zs_dopri)]

abw_eigen = jnp.array([uz_eigen[i,0] - f_analytical(zs_eigen[i]) for i in range(n)])
abw_dopri = jnp.array([uz_dopri[i,0] - f_analytical(zs_dopri[i]) for i in range(len(zs_dopri))])
abw_mean_eigen = jnp.mean(jnp.abs(abw_eigen))
abw_mean_dopri = jnp.mean(jnp.abs(abw_dopri))
abw_max_eigen = jnp.max(jnp.abs(abw_eigen))
abw_max_dopri = jnp.max(jnp.abs(abw_dopri))

df = pd.concat([df, pd.DataFrame({'abw_mean_eigen': [abw_mean_eigen], 'abw_mean_dopri': [abw_mean_dopri], 'abw_max_eigen': [abw_max_eigen], 'abw_max_dopri': [abw_max_dopri]})], ignore_index=True)

# Plot
fig, ax1 = plt.subplots(figsize=(20,10))
plt.title('Test')
ax1.set_xlabel('z/pc')
ax1.set_ylabel('$\Phi / (km/s)^2$')
ax1.scatter(zs_eigen, [u[0] for u in uz_eigen], marker='o', color = 'black', label = 'eigenerSolver')
ax1.scatter(zs_dopri, [u[0] for u in uz_dopri], marker='o', color = 'red', label = 'Dopri5')
zs = jnp.linspace(z0, z1, 100)
ax1.plot(zs, f_analytical(zs), color = 'blue', label = 'Analytische Lösung')
ax1.grid()
ax1.legend()
fig.tight_layout()

df.to_string('comparableparams.txt', col_space=16, justify='left', index=False)



eigenerSolverV2


AttributeError: 'NoneType' object has no attribute 'block_until_ready'

AttributeError: 'NoneType' object has no attribute 'block_until_ready'